# Modul 11: Metriken, Suche und erklärbare Merkmale | Übungen

## Überblick

Sie wählen Metriken passend zu Fehlerfolgen, führen leakage-sichere Kreuzvalidierung durch, optimieren kleine Suchräume und konstruieren oder wählen Merkmale innerhalb von Pipelines. Zum Abschluss erklären Sie Modellentscheidungen mit Koeffizienten und Permutationswichtigkeit.

**Zugehörige Vorlesungen**

- **Metriken und Suche**
- **Merkmale erklären**

## Lernziele

Nach der Bearbeitung können Sie:

- Klassifikations- und Regressionsmetriken einschließlich ROC- und Precision-Recall-Kurven fachlich auswählen.
- Kreuzvalidierung und Hyperparametersuche mit passenden Splittern leakage-sicher durchführen.
- Merkmalskonstruktion, Auswahl, Reduktion und Modellinterpretation in reproduzierbaren Pipelines verbinden.

## Geprüfte Fähigkeiten

- Schwellenanalyse, ROC-AUC, Average Precision, MAE und RMSE
- StratifiedKFold, GroupKFold, cross_validate, GridSearchCV und RandomizedSearchCV
- PolynomialFeatures, SelectKBest, PCA, Koeffizienten und permutation_importance

## Hinweise zur Bearbeitung

Dieses Notebook dient als praktische Übung und Lernstandskontrolle. Führen Sie zuerst die Einrichtungszelle aus und bearbeiten Sie danach die Aufgaben in der angegebenen Reihenfolge. Die vorgesehenen Arbeitsbereiche sind deutlich markiert.

- **Erwarteter Schwierigkeitsgrad:** fortgeschritten
- Verwenden Sie sprechende Variablennamen und prüfen Sie wichtige Zwischenformen und Wertebereiche.
- Verändern Sie die vorgegebenen Zufalls-Startwerte nur, wenn eine Aufgabe dies ausdrücklich verlangt.
- Interpretieren Sie Ergebnisse fachlich. Eine einzelne Kennzahl ist selten eine vollständige Begründung.
- Alle Aufgaben sind für die kostenlose Google-Colab-Umgebung ausgelegt. Die Datensätze und Modelle sind bewusst klein gehalten. Eine GPU ist nicht erforderlich, kann aber bei einzelnen Deep-Learning-Aufgaben die Laufzeit verkürzen.

## Einrichtung und gemeinsame Datenbasis

Die Setup-Zelle lädt den kleinen Brustkrebs-Datensatz aus scikit-learn und erzeugt zusätzlich Gruppenkennungen sowie einen kleinen nichtlinearen Regressionsdatensatz. Alle Daten bleiben lokal verfügbar.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split

RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)

krebs = load_breast_cancer(as_frame=True)
X_klassifikation = krebs.data.copy()
y_klassifikation = krebs.target.copy()

X_train, X_test, y_train, y_test = train_test_split(
    X_klassifikation,
    y_klassifikation,
    test_size=0.25,
    stratify=y_klassifikation,
    random_state=RANDOM_SEED,
)

# Künstliche Gruppen simulieren mehrere Messungen aus derselben Quelle.
gruppen = np.repeat(np.arange(0, int(np.ceil(len(X_klassifikation) / 4))), 4)[: len(X_klassifikation)]

# Kleiner nichtlinearer Regressionsdatensatz für Merkmalskonstruktion.
x_reg = np.linspace(-3.0, 3.0, 180)
y_reg = 2.0 + 1.2 * x_reg - 0.9 * x_reg**2 + rng.normal(0.0, 0.8, size=x_reg.size)
X_reg = x_reg.reshape(-1, 1)

print("Klassifikationsdaten:", X_klassifikation.shape)
print("Regressionsdaten:", X_reg.shape)

### Aufgabe 1: Schwellenwerte und Fehlerfolgen untersuchen

Trainieren Sie eine skalierte logistische Regression. Berechnen Sie für die Schwellenwerte `0.30`, `0.50` und `0.70` jeweils Accuracy, Precision, Recall und F1. Stellen Sie die Ergebnisse tabellarisch dar und bestimmen Sie den Schwellenwert mit dem höchsten Recall.

Begründen Sie anschließend, warum ein hoher Recall in einer medizinischen Screening-Situation wichtig sein kann und welche Gegenleistung dafür häufig entsteht.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# ============================================================
# IHRE LÖSUNG HIER
# ============================================================

> **Ihre Antwort:**
>
> Erklären Sie den Zielkonflikt zwischen Recall und Precision in diesem Kontext.

### Aufgabe 2: ROC- und Precision-Recall-Kurven vergleichen

Berechnen und visualisieren Sie für das trainierte Modell eine ROC-Kurve und eine Precision-Recall-Kurve. Geben Sie ROC-AUC und Average Precision aus. Markieren Sie in beiden Diagrammen die Leistung einer zufälligen beziehungsweise konstanten Referenz.

In [ ]:
from sklearn.metrics import (
    RocCurveDisplay,
    PrecisionRecallDisplay,
    roc_auc_score,
    average_precision_score,
)

# ============================================================
# IHRE LÖSUNG HIER
# ============================================================

> **Ihre Antwort:**
>
> Wann ist die Precision-Recall-Kurve meist aussagekräftiger als die ROC-Kurve?

### Aufgabe 3: Kreuzvalidierungsstrategien leakage-sicher prüfen

Vergleichen Sie für dieselbe Pipeline drei Bewertungsvarianten:

1. `StratifiedKFold` mit fünf Folds,
2. `GroupKFold` mit fünf Folds und den vorgegebenen Gruppen,
3. einen absichtlich ungeeigneten Split, bei dem die vollständigen Daten vor der Kreuzvalidierung skaliert werden.

Berechnen Sie jeweils den mittleren F1-Wert. Erklären Sie, warum Variante 3 methodisch falsch ist, auch wenn ihre Kennzahl nicht immer deutlich höher ausfällt.

In [ ]:
from sklearn.model_selection import StratifiedKFold, GroupKFold, cross_val_score

# ============================================================
# IHRE LÖSUNG HIER
# ============================================================

> **Ihre Antwort:**
>
> Warum muss Vorverarbeitung innerhalb einer Pipeline liegen?

### Aufgabe 4: Kleine Hyperparametersuche dokumentieren

Führen Sie eine `GridSearchCV` für eine Pipeline aus Standardisierung und logistischer Regression durch. Prüfen Sie ausschließlich die Werte `C = [0.01, 0.1, 1.0, 10.0]`. Nutzen Sie stratifizierte fünfteilige Kreuzvalidierung und F1 als Zielmetrik.

Erstellen Sie aus `cv_results_` eine kompakte Tabelle mit Parameterwert, mittlerem F1-Wert, Standardabweichung und Rang. Bewerten Sie das beste Modell einmalig auf dem unberührten Testdatensatz.

In [ ]:
from sklearn.model_selection import GridSearchCV

# ============================================================
# IHRE LÖSUNG HIER
# ============================================================

> **Ihre Antwort:**
>
> Warum darf der Testdatensatz nicht zur Wahl von C verwendet werden?

### Aufgabe 5: Nichtlineare Merkmale und Regularisierung verbinden

Teilen Sie den Regressionsdatensatz reproduzierbar auf. Vergleichen Sie:

- eine lineare Ridge-Regression nur mit `x`,
- eine Pipeline aus `PolynomialFeatures(degree=2)`, Standardisierung und Ridge.

Berechnen Sie RMSE und R² auf dem Testdatensatz und zeichnen Sie die Vorhersagekurven über einem geordneten Gitter. Diskutieren Sie, weshalb der Polynomgrad innerhalb der Pipeline erzeugt werden sollte.

In [ ]:
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import PolynomialFeatures

# ============================================================
# IHRE LÖSUNG HIER
# ============================================================

> **Ihre Antwort:**
>
> Welche Gefahr entsteht bei sehr hohen Polynomgraden?

### Aufgabe 6: Integrationsaufgabe: Auswahl und Interpretation in einer Pipeline

Bauen Sie eine Pipeline aus Standardisierung, `SelectKBest(f_classif, k=10)` und logistischer Regression. Trainieren Sie sie auf den Trainingsdaten und bewerten Sie F1 auf dem Testdatensatz.

Ermitteln Sie anschließend:

1. die zehn ausgewählten Originalmerkmale,
2. die standardisierten Koeffizienten des Klassifikators,
3. die fünf wichtigsten Testmerkmale nach Permutationswichtigkeit.

Erklären Sie, warum Koeffizienten und Permutationswichtigkeit nicht zwingend dieselbe Rangfolge liefern.

In [ ]:
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.inspection import permutation_importance

# ============================================================
# IHRE LÖSUNG HIER
# ============================================================

> **Ihre Antwort:**
>
> Warum können beide Erklärungsverfahren zu unterschiedlichen Rangfolgen kommen?

## Abschlusskontrolle

Prüfen Sie vor dem Abschluss:

- Lassen sich alle Zellen in sinnvoller Reihenfolge ausführen?
- Sind Formen, Datentypen, Wertebereiche und Zufalls-Startwerte dokumentiert?
- Wurden Trainings-, Validierungs- und Testinformationen sauber getrennt?
- Sind Diagramme und Kennzahlen beschriftet und fachlich interpretiert?
- Können Sie erklären, warum die gewählten Methoden zur Aufgabenstellung passen?